In [ ]:
%load_ext autoreload
%autoreload 2

params = {
    "N" : 10,
    "K" : 2,
    "B_train" : 50,
    "B_test" : 100, 
    "tau" : 0.05,
    "sigma_u" : 1.0,  
    "warmup_steps" : 100,
    "num_samples" : 150,
    "target_accept_prob" : 0.75,
    "seed" : 42
}
import torch
import numpy as np
import pyro
import pyro.infer as infer
from pyro.infer import MCMC, NUTS

def enforce_reproducibility(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    pyro.clear_param_store()
    pyro.set_rng_seed(seed)

enforce_reproducibility(params["seed"])
print(f"✅ Workspace locked. Ready for configuration setup.")

In [ ]:
import os
import sys
#current folder path
current_notebook_dir = os.getcwd()

#find parent folder one level up
project_root_dir = os.path.abspath(os.path.join(current_notebook_dir, "../.."))

if project_root_dir not in sys.path:
    sys.path.append(project_root_dir)
    print(f"Successfully added project root to path: {project_root_dir}")

from src.model import PhylogeneticPrior
from src.diagnostics import (
    evaluate_test_diagnostics,
    print_diagnostic_report,
    compute_nj_residual
)
from src.penalty import get_fresh_test_quartets


In [ ]:
#guassian prior
# prior model represents the base object, the framework
prior_model = PhylogeneticPrior(
    N= params ["N"],
    K=params ["K"],
    B= params["B_train"],
    sigma_u=params["sigma_u"],
    seed = params["seed"]
)
print("Prior model parameters are:", prior_model)

#fresh test quartets for validation from function in penalty file
test_quartets = get_fresh_test_quartets(
    N= params ["N"],
    B_test = params["B_test"],
    train_quartets=prior_model.fixed_indices,
    seed = 123
)




In [ ]:
#lambda grid for test run
lambda_grid = [0.0, 0.1, 0.5,1.0, 2.5, 5.0]
results ={} #dictionary to store results

In [ ]:
#Running test run , for every value of lambda in the grid
for lmbda in lambda_grid:
    print(f" EXPERIMENT RUN: λ4 = {lmbda} (N={params['N']})")

    # conditioned model is the model with specific hyperparameters
    # creates specific model instance 
    def model_conditioned():
        """this model returns specific model instance with value of hyperparameters."""
        return prior_model.initialize(
            lmbda_4 = lmbda,
            lmbda_g = 0.0,
            tau= params ["tau"],
            use_scale= False #to isolate lambda sweep
        )
    
    enforce_reproducibility(params["seed"])

    #building NUTS engine
    nuts_kernel = NUTS(model_conditioned, adapt_step_size=True, target_accept_prob=params["target_accept_prob"] )
    mcmc =  MCMC(
        kernel = nuts_kernel,
        num_samples= params["num_samples"],
        warmup_steps=params["warmup_steps"],
        num_chains= 1
    )

    #running chains
    mcmc.run()
    posterior_samples = mcmc.get_samples()
    
    #we only got posterior samples : latent points, not their distance matrices yet
    D_samples_list = []

    #we get 150 samples as we asked- coordinates of these 150 samples
    #posterior samples : [150,10,2] i.e. total 150 samples, for 10 taxa and 2 dimensions
    #checking no of samples collected by MCMC
    num_samples_collected = posterior_samples[next(iter(posterior_samples))].shape[0]
 
    for s in range(num_samples_collected):
        single_sample = {k: v[s:s+1] for k, v in posterior_samples.items()} 
        try:
            # Replay each individual sample configuration separately
            predictive = infer.Predictive(model_conditioned, posterior_samples=single_sample)
            single_pred = predictive()
            D_matrix = single_pred["D_tilde"].squeeze(0)
            
            if torch.isnan(D_matrix).any() or torch.isinf(D_matrix).any():
                continue
                
            D_samples_list.append(D_matrix.detach().cpu().numpy())
        except Exception:
            continue

    if len(D_samples_list) == 0:
        print(f"❌ Landscape Failure: Complete numerical overflow at λ4 = {lmbda}")
        results[lmbda] = {"metrics": None, "mean_nj_residual": np.nan, "divergences": "ALL"}
        continue
        
    D_samples = np.stack(D_samples_list)


model_conditioned

    ↓

defines probability distribution

NUTS

    ↓

defines transition dynamics

MCMC

    ↓
    
runs the sampling process